<a href="https://colab.research.google.com/github/karanbisht-kb/AIML/blob/main/GoldAss-ImageNet-pretrained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


In [5]:

# ImageNet normalization

transform = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}

data_dir = "/content/dataset"  # change path




def is_valid_file(x):
    return x.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))

train_dataset = datasets.ImageFolder(
    os.path.join(data_dir, 'train'),
    transform=transform['train'],
    is_valid_file=is_valid_file
)

val_dataset = datasets.ImageFolder(
    os.path.join(data_dir, 'val'),
    transform=transform['val'],
    is_valid_file=is_valid_file
)

#train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform['train'])
#val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform['val'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/train'

In [ ]:

model = models.resnet18(pretrained=True)

# Freeze all layers (optional)
for param in model.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

# Unfreeze last layer
for param in model.fc.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [ ]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [ ]:

def train_model(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()

        train_acc = correct / len(train_dataset)

        # Validation
        model.eval()
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()

        val_acc = val_correct / len(val_dataset)

        print(f"Epoch {epoch+1}/{epochs} "
              f"Loss: {running_loss:.4f} "
              f"Train Acc: {train_acc:.4f} "
              f"Val Acc: {val_acc:.4f}")

train_model(model, train_loader, val_loader, epochs=5)


In [ ]:

from PIL import Image

def predict_image(image_path):
    model.eval()

    image = Image.open(image_path).convert('RGB')
    image = transform['val'](image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Example
print(predict_image("/content/test.jpg"))
